---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: AI and Data-Driven Marketplaces

### 📋 **Topic**: Analyzing and Visualizing Experiment Results

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---


> **Before you begin:** [Update your course files and Python environment safely.](https://github.com/apostolosfilippas/ma/blob/main/assets/before-you-begin.md)

## Overview

Let's use our Python knowledge to analyze and visualize the results of an experiment.

**Experiment Setup:**
- Control group: "status quo" pricing algorithm
- Treatment group: "new" pricing algorithm that we want to evaluate  
- Outcome: Earnings at the end of a three month period

**What we'll learn:**
- How to analyze experimental results
- Statistical comparison between groups
- Visualization of treatment effects
- Understanding experimental outcomes


In [ ]:
# Let's import the libraries we will use
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Locate the course folder from either ma/ or ma/scripts/.
course_folder = Path.cwd()
if not (course_folder / "data").is_dir():
    course_folder = course_folder.parent

data_folder = course_folder / "data"
output_folder = course_folder / "temp"
output_folder.mkdir(parents=True, exist_ok=True)

# Keep the group order and colors consistent with Lecture 7.
group_order = ["Control", "Treatment"]
group_colors = {"Control": "#1f77b4", "Treatment": "#ff7f0e"}
plt.rcParams["text.parse_math"] = False  # Show dollar signs as currency.

# Load experiment data.
df_users = pd.read_csv(data_folder / "earnings.csv")

print("Dataset loaded successfully!")
print(f"Dataset shape: {df_users.shape}")
print(f"Columns: {df_users.columns.tolist()}")

In [ ]:
# Show the same sample each time we run the notebook.
print(df_users.sample(20, random_state=42))

### Simple visualization of distributions of earnings for control and treatment groups

In [ ]:
# Compare the earnings distributions using the same bins for both groups.
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(
    data=df_users, x="earnings", hue="treatment",
    hue_order=group_order, palette=group_colors,
    alpha=0.45, bins=30, common_bins=True, kde=True, ax=ax,
)
ax.set_xlabel("Earnings over three months (USD)")
ax.set_ylabel("Number of users")
ax.set_title("Earnings distribution by treatment group")
sns.move_legend(ax, "upper right", title="Group")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(output_folder / "earnings_distribution.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

print("Earnings distribution saved to temp/earnings_distribution.pdf")

### Simple treatment effect calculation


In [ ]:
# Compare earnings between treatment and control groups.
# Keep full precision for calculations; round only the displayed table.
df_stats = df_users.groupby("treatment").agg({"earnings": ["mean", "var", "count"]})

# Flatten column names and keep Control first.
df_stats.columns = ["sample_mean", "sample_var", "sample_size"]
df_stats = df_stats.reindex(group_order).reset_index()

# Standard error measures the uncertainty in each group's estimated mean.
df_stats["sample_se"] = np.sqrt(df_stats["sample_var"] / df_stats["sample_size"])

print("Detailed statistics by treatment group:")
print(df_stats.round(2))

# The estimated treatment effect is the difference in average earnings.
treatment_mean = df_stats.loc[df_stats["treatment"] == "Treatment", "sample_mean"].iloc[0]
control_mean = df_stats.loc[df_stats["treatment"] == "Control", "sample_mean"].iloc[0]
treatment_effect = treatment_mean - control_mean
pct_change = treatment_effect / control_mean * 100

print(f"\nTreatment group average: ${treatment_mean:.2f}")
print(f"Control group average: ${control_mean:.2f}")
print(f"Estimated treatment effect: ${treatment_effect:.2f} per user ({pct_change:+.1f}%)")

### Confidence intervals for the group means

A mean is an estimate. Its **standard error** describes how much that estimate would vary across repeated experiments.

With large groups, an approximate 95% confidence interval is **mean ± 1.96 × standard error**. Each interval below is for one group's mean earnings, not for the treatment effect.

In [ ]:
# Show each group's mean and approximate 95% confidence interval.
fig, ax = plt.subplots(figsize=(8, 6))
for position, group in enumerate(group_order):
    row = df_stats.loc[df_stats["treatment"] == group].iloc[0]
    ax.errorbar(
        x=position, y=row["sample_mean"], yerr=1.96 * row["sample_se"],
        fmt="o" if group == "Control" else "s",
        capsize=5, elinewidth=2, markersize=8, color=group_colors[group],
    )

ax.set_xticks(range(len(group_order)), group_order)
ax.set_xlabel("Experimental group")
ax.set_ylabel("Mean earnings over three months (USD)")
ax.set_title("Group mean earnings with approximate 95% confidence intervals", pad=12)
ax.grid(True, alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(output_folder / "earnings_point_estimates.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

print("Group means saved to temp/earnings_point_estimates.pdf")

### Confidence interval for the treatment effect

Our question is about the **difference** between the means. For independent groups, add the two variances of the estimated means, then take the square root to get the difference's standard error.

The visualization below shows the estimated difference and its approximate 95% confidence interval. Zero means no average effect. Compare this interval with zero rather than judging significance from overlap between the two group intervals.


In [ ]:
# Combine uncertainty from the two independent groups.
se_diff = np.sqrt((df_stats["sample_var"] / df_stats["sample_size"]).sum())
ci_lower = treatment_effect - 1.96 * se_diff
ci_upper = treatment_effect + 1.96 * se_diff

print(f"Estimated effect per user: ${treatment_effect:.2f}")
print(f"Approximate 95% confidence interval: [${ci_lower:.2f}, ${ci_upper:.2f}]")

# Plot the difference itself, with zero as the reference point.
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.axvline(0, color="0.4", linestyle="--", linewidth=1)
ax.errorbar(
    x=treatment_effect, y=0, xerr=1.96 * se_diff,
    fmt="o", color="#1f77b4", markersize=8, capsize=6, elinewidth=2,
)
ax.text(
    treatment_effect, 0.16,
    f"${treatment_effect:.2f} per user\n95% CI: [${ci_lower:.2f}, ${ci_upper:.2f}]",
    ha="center", va="bottom",
)
ax.set_xlim(min(0, ci_lower) - 5, max(0, ci_upper) + 5)
ax.set_ylim(-0.5, 0.5)
ax.set_yticks([])
ax.set_xlabel("Difference in mean three-month earnings: treatment minus control (USD)")
ax.set_title("Estimated treatment effect with approximate 95% confidence interval", pad=12)
ax.grid(True, alpha=0.3, axis="x")
fig.tight_layout()
fig.savefig(output_folder / "earnings_treatment_effect.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)


## (Optional) Statistical significance testing

A two-sided p-value asks how unusual a difference at least this extreme would be if the true difference were zero. It is not the probability that the treatment has no effect.

We'll use Welch's t-test, which allows the two groups to have different variances. The confidence interval above uses a large-sample normal approximation.

In [ ]:
# Compare the group means with Welch's two-sample t-test.
from scipy import stats

treatment_earnings = df_users.loc[df_users["treatment"] == "Treatment", "earnings"]
control_earnings = df_users.loc[df_users["treatment"] == "Control", "earnings"]
ttest_result = stats.ttest_ind(treatment_earnings, control_earnings, equal_var=False)

print(f"T-statistic: {ttest_result.statistic:.4f}")
# Scientific notation shows a very small p-value without rounding it to zero.
print(f"P-value: {ttest_result.pvalue:.3e}")

# Cohen's d expresses the difference in standard-deviation units.
pooled_std = np.sqrt(
    (
        (len(treatment_earnings) - 1) * treatment_earnings.var()
        + (len(control_earnings) - 1) * control_earnings.var()
    )
    / (len(treatment_earnings) + len(control_earnings) - 2)
)
cohens_d = treatment_effect / pooled_std
print(f"Cohen's d: {cohens_d:.4f}")

In [ ]:
# A box plot gives another view of the earnings distributions.
fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(
    data=df_users, x="treatment", y="earnings", order=group_order,
    hue="treatment", hue_order=group_order, palette=group_colors,
    legend=False, ax=ax,
)
ax.set_xlabel("Experimental group")
ax.set_ylabel("Earnings over three months (USD)")
ax.set_title("Earnings distribution by treatment group")
ax.grid(True, alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(output_folder / "earnings_boxplot.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

print("Box plot saved to temp/earnings_boxplot.pdf")

# Report uncertainty and magnitude separately.
significance_level = 0.05
is_significant = ttest_result.pvalue < significance_level

print(f"\nP-value: {ttest_result.pvalue:.3e}")
print(f"Statistically significant at the 5% level: {'Yes' if is_significant else 'No'}")
print(f"Estimated effect per user: ${treatment_effect:.2f} ({pct_change:+.1f}%)")
print(f"Approximate 95% confidence interval: [${ci_lower:.2f}, ${ci_upper:.2f}]")
print(f"Cohen's d: {cohens_d:.4f}")

if ci_lower > 0:
    print("The interval is entirely above zero: the results support a positive average effect.")
elif ci_upper < 0:
    print("The interval is entirely below zero: the results support a negative average effect.")
else:
    print("The interval includes zero: these results do not establish the direction of the average effect.")

print("Whether the effect is worth pursuing also depends on costs and other marketplace outcomes.")